In [23]:
!uv pip install --upgrade pip

# Uninstall conflicting packages
!uv pip uninstall langchain-core langchain-openai langchain-experimental langchain-community langchain chromadb beautifulsoup4 python-dotenv PyPDF2 rank_bm25

# Install compatible versions of langchain-core and langchain-openai
!uv pip install langchain-core==0.3.6
!uv pip install langchain-openai==0.2.1
!uv pip install langchain-experimental==0.3.2
!uv pip install langchain-community==0.3.1
!uv pip install langchain==0.3.1

# Install remaining packages
!uv pip install chromadb==0.5.11
!uv pip install python-dotenv==1.0.1
!uv pip install PyPDF2==3.0.1
!uv pip install rank_bm25==0.2.2


# Restart the kernel after installation if needed

Using Python 3.12.13 environment at: /usr
Resolved 1 package in 23ms
Checked 1 package in 0.07ms
Using Python 3.12.13 environment at: /usr
Uninstalled 10 packages in 153ms
 - beautifulsoup4==4.13.5
 - chromadb==0.5.11
 - langchain==0.3.1
 - langchain-community==0.3.1
 - langchain-core==0.3.63
 - langchain-experimental==0.3.2
 - langchain-openai==0.2.1
 - pypdf2==3.0.1
 - python-dotenv==1.0.1
 - rank-bm25==0.2.2
Using Python 3.12.13 environment at: /usr
Resolved 23 packages in 182ms
Prepared 1 package in 30ms
Installed 1 package in 6ms
 + langchain-core==0.3.6
Using Python 3.12.13 environment at: /usr
Resolved 31 packages in 102ms
Prepared 1 package in 13ms
Installed 1 package in 4ms
 + langchain-openai==0.2.1
Using Python 3.12.13 environment at: /usr
Resolved 48 packages in 490ms
Prepared 7 packages in 144ms
Uninstalled 3 packages in 4ms
Installed 7 packages in 31ms
 + langchain==0.3.28
 + langchain-community==0.3.31
 - langchain-core==0.3.6
 + langchain-core==0.3.83
 + langchain-exper

In [24]:
# You can view installed packages and versions using the statement below
!uv pip list

Using Python 3.12.13 environment at: /usr
Package                                  Version
---------------------------------------- -----------------
absl-py                                  1.4.0
accelerate                               1.13.0
aiofiles                                 25.1.0
aiohappyeyeballs                         2.6.1
aiohttp                                  3.13.3
aiosignal                                1.4.0
altair                                   5.5.0
annotated-doc                            0.0.4
annotated-types                          0.7.0
anyio                                    4.12.1
anywidget                                0.9.21
apsw                                     3.51.3.0
apswutils                                0.1.2
argon2-cffi                              25.1.0
argon2-cffi-bindings                     25.1.0
array-record                             0.8.3
arrow                                    1.4.0
asgiref                                  

Step 1: Import Libraries

In [25]:
import os
os.environ['USER_AGENT'] = 'RAGUserAgent'
import openai
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

#langchain packages
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

import chromadb
from langchain_community.vectorstores import Chroma
from langchain_core.runnables import RunnableParallel
from dotenv import load_dotenv, find_dotenv
from langchain_core.prompts import PromptTemplate
from PyPDF2 import PdfReader

from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain_community.retrievers import BM25Retriever

# EnsembleRetriever
from langchain.retrievers import EnsembleRetriever

Step 2: Import API Keys - Langsmith & OpenAI

In [32]:
from google.colab import userdata

os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')
openai.api_key = os.environ['OPENAI_API_KEY']

os.environ["LANGSMITH_API_KEY"] = userdata.get("LANGSMITH_API_KEY")
langsmith_api_key = os.environ["LANGSMITH_API_KEY"]

pdf_path = "/Value Add Real Estate.pdf"
collection_name = "value_add_real_estate"

Step 3: Define LLM Model, embedding function and user query

In [33]:
# Declare LLM Model, embedding_function and user_query

llm = ChatOpenAI(model_name = "gpt-4o", temperature=0)
user_query = "What strategies do value-add managers use to protect against downside risk?"

str_output_parser = StrOutputParser()

#embedding function
embedding_function = OpenAIEmbeddings()

Step 4: Reading the pdf and splitting the content.

In [34]:
# Load the PDF and extract text
pdf_reader = PdfReader(pdf_path)
text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

In [35]:
# Split
character_splitter = RecursiveCharacterTextSplitter(
    separators=["\n\n", "\n", ". ", " ", ""],
    chunk_size=1000,
    chunk_overlap=200
)
splits = character_splitter.split_text(text)

In [36]:
dense_documents = [Document(page_content=text, metadata={"id": str(i), "source": "dense"}) for i, text in enumerate(splits)]
sparse_documents = [Document(page_content=text, metadata={"id": str(i), "source": "sparse"}) for i, text in enumerate(splits)]

Step 5: Create vectorstore using chroma

In [37]:
chroma_client = chromadb.Client()

vectorstore = Chroma.from_documents(
    documents=dense_documents,
    embedding=embedding_function,
    collection_name=collection_name,
    client=chroma_client
)

ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
ERROR:chromadb.telemetry.product.posthog:Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Step 6: Create Dense, Sparse, and Ensemble Retrievers

In [38]:
# Create dense retriever
dense_retriever = vectorstore.as_retriever(search_kwargs={"k": 10})

# Create sparse retriever
sparse_retriever = BM25Retriever.from_documents(sparse_documents, k=10)

# initialize the ensemble retriever
ensemble_retriever = EnsembleRetriever(retrievers=[dense_retriever, sparse_retriever], weights=[0.5, 0.5], c=0)

Step 7: Pull prompt from Langsmith

In [39]:
# Here we use latest prompt template from Langchain hub which has been updated since the author wrote his
# Notebook. The author's version is now deprecated. We are using the latest packages from langsmith. You will need
#Langsmith api_key to be passed to Client() object here.
from langsmith import Client
from langchain_openai import ChatOpenAI

client = Client(api_key=langsmith_api_key)
prompt = client.pull_prompt("jclemens24/rag-prompt")

In [40]:
# Relevance check and Prompt Template
relevance_prompt_template = PromptTemplate.from_template(
    """
    Given the following question and retrieved context, determine if the context is relevant to the question.
    Provide a score from 1 to 5, where 1 is not at all relevant and 5 is highly relevant.
    Return ONLY the numeric score, without any additional text or explanation.

    Question: {question}
    Retrieved Context: {retrieved_context}

    Relevance Score:"""
)

Step 8: Post processing and extracting

In [41]:
# Post-processing
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

In [42]:
def extract_score(llm_output):
    try:
        score = float(llm_output.strip())
        return score
    except ValueError:
        return 0

# Chain it all together with LangChain
def conditional_answer(x):
    relevance_score = extract_score(x['relevance_score'])
    if relevance_score < 4:
        return "I don't know."
    else:
        return x['answer']

Step 9: Defining RAG chain for documents

In [43]:
rag_chain_from_docs = (
    RunnablePassthrough.assign(context=(lambda x: format_docs(x["context"])))
    | RunnableParallel(
        {"relevance_score": (
            RunnablePassthrough()
            | (lambda x: relevance_prompt_template.format(question=x['question'], retrieved_context=x['context']))
            | llm
            | str_output_parser
        ), "answer": (
            RunnablePassthrough()
            | prompt
            | llm
            | str_output_parser
        )}
    )
    | RunnablePassthrough().assign(final_answer=conditional_answer)
)

Step 10: Defining RAG with Source Documents using EnsembleRetriever

In [44]:
rag_chain_with_source = RunnableParallel(
    {"context": ensemble_retriever, "question": RunnablePassthrough()}
).assign(answer=rag_chain_from_docs)

Step 11: Obtaining result and answers after passing the user query

In [45]:
# Pass User_Query Here
result = rag_chain_with_source.invoke(user_query)
retrieved_docs = result['context']

# Print output and answer to the query
print(f"Original Question: {user_query}\n")
print(f"Relevance Score: {result['answer']['relevance_score']}\n")
print(f"Final Answer:\n{result['answer']['final_answer']}\n\n")
print("Retrieved Documents:")
for i, doc in enumerate(retrieved_docs, start=1):
    print(f"Document {i}: Document ID: {doc.metadata['id']} source: {doc.metadata['source']}")
    print(f"Content:\n{doc.page_content}\n")

Original Question: What strategies do value-add managers use to protect against downside risk?

Relevance Score: 5

Final Answer:
Value-add managers use several strategies to protect against downside risk:

1. **Investing in Lower Volatility Property Sectors**: They focus on property sectors that exhibit less volatility, such as the multifamily sector, which tends to have lower volatility in rent and occupancy levels.

2. **Acquiring Quality Assets with Durable In-Place Income**: Managers seek properties that provide steady streams of income, which can help offset the volatility associated with real estate price movements.

3. **Limiting Leverage**: By using modest leverage, managers aim to reduce financial risk and enhance the stability of returns.

4. **Gaining Portfolio Diversification**: Diversifying across property sectors and geographies helps mitigate the risk of poor performance from any single source.

These strategies collectively aim to provide defensive characteristics in t